<a href="https://colab.research.google.com/github/AlvarFeher/TFG/blob/main/LetsEndThis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
filename = "/content/drive/MyDrive/TFG/data/filtered_data_good.pkl"
!pip install awkward pandas awkward-pandas

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import pickle
import networkx as nx

# Cargar el dataset
with open(filename, 'rb') as file:
    filtered_df = pickle.load(file)

# Asegurar que las columnas sean listas de Python si vienen en awkward arrays
filtered_df['digitE'] = filtered_df['digitE'].apply(lambda x: x[0] if isinstance(x, list) else x)
filtered_df['digitR'] = filtered_df['digitR'].apply(lambda x: x[0] if isinstance(x, list) else x)
filtered_df['digitC'] = filtered_df['digitC'].apply(lambda x: x[0] if isinstance(x, list) else x)
filtered_df['MCID'] = filtered_df['MCID'].apply(lambda x: x[0] if isinstance(x, list) else x)

# Ahora estamos incluyendo TODAS las partículas, sin filtrar solo fotones
particles = filtered_df.copy()

print(f"Total de particulas en el dataset: {len(particles)}")
print("Ejemplo de datos:")
print(particles.head())

Total de particulas en el dataset: 13615740
Ejemplo de datos:
                            evtnum  digitE  digitR  digitC   MCID       MCP
entry subentry subsubentry                                                 
0     0        0            303626   5.538    12.0     2.0  130.0  4.110076
               1            303626   5.538    12.0     2.0  130.0  4.110076
               2            303626   5.538    12.0     2.0  130.0  4.110076
               3            303626   5.538    12.0     2.0  130.0  4.110076
               4            303626   5.538    12.0     2.0  130.0  4.110076


In [ ]:
def find_seeds_optimized(particles, energy_threshold=50):
    seeds = []
    particle_dict = {(int(r['digitR']), int(r['digitC'])): (i, r['digitE'], r['MCID'])
                     for i, r in particles.iterrows()}

    for (r, c), (idx, energy, mcid) in particle_dict.items():
        if energy < energy_threshold:
            continue

        # Get the 8-connected neighborhood
        is_local_max = True
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                neighbor = (r + dr, c + dc)
                if neighbor in particle_dict:
                    if particle_dict[neighbor][1] >= energy:
                        is_local_max = False
                        break
            if not is_local_max:
                break

        if is_local_max:
            seeds.append((idx, r, c, energy, mcid))

    return seeds

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from scipy.spatial import cKDTree

def process_event(particles, energy_threshold=50, window_size=5):
    seeds = find_seeds_optimized(particles, energy_threshold=energy_threshold)

    if not seeds:
        return None

    clusters = extract_fixed_window_clusters(particles, seeds, window_size=window_size)

    cluster_analysis = []
    clustered_nodes = {}

    for cluster_id, cluster in enumerate(clusters):
        cluster_info = {
            "Cluster ID": cluster_id,
            "Seed ID": cluster['seed_id'],
            "Seed Row": cluster['seed_row'],
            "Seed Column": cluster['seed_col'],
            "Total Energy": cluster['total_energy'],
            "Seed_MCID": cluster['mcid']
        }

        cluster_analysis.append(cluster_info)

        clustered_nodes[cluster['seed_id']] = {
            'node_id': cluster['seed_id'],
            'row': cluster['seed_row'],
            'column': cluster['seed_col'],
            'energy': cluster['total_energy'],
            'MCID': cluster['mcid'],
            'type': 'seed',
            'clusterId': cluster_id
        }

    return cluster_analysis, clustered_nodes, None  # 'None' placeholder for graph G if not used


In [ ]:
# -------------------------------
# Other functions remain mostly unchanged.
# You can keep build_graph, update_overlapping_edges, analyze_clusters as before.
# -------------------------------

def build_graph(particles, seeds, max_distance=2, min_energy_ratio=0.05):
    G = nx.DiGraph()

    # Add seed nodes
    for seed in seeds:
        idx, r, c, energy, mcid = seed
        G.add_node(idx, row=r, column=c, energy=energy, type='seed', MCID=mcid)

    # Add neighbor nodes (non-seeds) with constraints
    for index, row in particles.iterrows():
        if index in [s[0] for s in seeds]:
            continue

        r, c, e = row['digitR'], row['digitC'], row['digitE']
        valid_seeds = []
        for seed in seeds:
            s_idx, sr, sc, s_energy, s_mcid = seed
            distance = abs(sr - r) + abs(sc - c)
            if distance <= max_distance and e / s_energy >= min_energy_ratio:
                valid_seeds.append((seed, distance))

        if not valid_seeds:
            continue  # Skip this node if no nearby valid seed

        # Choose closest seed among valid options
        closest_seed, _ = min(valid_seeds, key=lambda x: x[1])
        s_idx, sr, sc, s_energy, s_mcid = closest_seed

        G.add_node(index, row=r, column=c, energy=e, type='neighbor', MCID=row['MCID'])
        G.add_edge(index, s_idx, weight=e / s_energy)

    return G


In [ ]:
def update_overlapping_edges(G):
    for node in G.nodes:
        in_edges = list(G.in_edges(node, data=True))
        if len(in_edges) > 1:
            total_energy = sum(G.nodes[src]['energy'] for src, _, _ in in_edges)
            for src, _, edge_data in in_edges:
                seed_energy = G.nodes[src]['energy']
                weight = seed_energy / total_energy if total_energy > 0 else 0
                edge_data['weight'] = weight
    overlap_nodes = [node for node in G.nodes if len(list(G.in_edges(node))) > 1]
    return G, overlap_nodes

printear el progreso, por q evento va los datos q genera...


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

def analyze_clusters(G):
    """
    Find weakly connected components (clusters) in the directed graph G,
    gather cluster metadata, and plot a heatmap for each cluster.

    Parameters
    ----------
    G : nx.DiGraph or nx.MultiDiGraph
        A NetworkX graph whose nodes have attributes:
          - 'type': str, e.g. 'seed' or something else
          - 'MCID': any identifier
          - 'energy': float
          - 'row': int
          - 'column': int

    Returns
    -------
    cluster_analysis : list of dict
        Information about each cluster (size, seed, dominant MCID, etc.)
    clustered_nodes : dict
        Key: cluster_id, Value: list of node-attribute dicts
    """
    clusters = list(nx.weakly_connected_components(G))
    cluster_analysis = []
    clustered_nodes = {}

    for cluster_id, cluster in enumerate(clusters):
        # Convert from set to list for indexing
        cluster_nodes = list(cluster)

        # We only proceed if there is exactly one seed in this cluster
        cluster_seeds = [node for node in cluster_nodes if G.nodes[node]['type'] == 'seed']
        if len(cluster_seeds) != 1:
            continue

        seed_id = cluster_seeds[0]
        seed_mcid = G.nodes[seed_id]['MCID']
        seed_energy = G.nodes[seed_id]['energy']

        # Collect energies by MCID to find the "dominant" MCID
        energy_by_particle = {}
        for node in cluster_nodes:
            mcid = G.nodes[node]['MCID']
            energy = G.nodes[node]['energy']
            energy_by_particle[mcid] = energy_by_particle.get(mcid, 0) + energy

        dominant_particle = max(energy_by_particle, key=energy_by_particle.get)

        # Build a list/dict of node attributes for easy reference
        clustered_nodes[cluster_id] = [
            {
                'node_id': node,
                'energy': G.nodes[node]['energy'],
                'MCID': G.nodes[node]['MCID'],
                'type': G.nodes[node]['type'],
                'row': G.nodes[node]['row'],
                'column': G.nodes[node]['column'],
                'clusterId': cluster_id
            }
            for node in cluster_nodes
        ]

        # Store summary info about this cluster
        cluster_analysis.append({
            'Cluster_ID': cluster_id,
            'Cluster_Size': len(cluster_nodes),
            'Seed_MCID': seed_mcid,
            'Seed_Energy': seed_energy,
            'Dominant_MCID': dominant_particle,
            'Dominant_Energy': energy_by_particle[dominant_particle]
        })

        # ------------------------------------------------------------
        # Create and plot a heatmap for this cluster
        # ------------------------------------------------------------
        # 1) Determine the bounding box (min/max row/col)
        min_row = min(G.nodes[n]['row'] for n in cluster_nodes)
        max_row = max(G.nodes[n]['row'] for n in cluster_nodes)
        min_col = min(G.nodes[n]['column'] for n in cluster_nodes)
        max_col = max(G.nodes[n]['column'] for n in cluster_nodes)

        # Cast to int explicitly
        height = int(max_row - min_row + 1)
        width  = int(max_col - min_col + 1)

        # Initialize the data array with integer dimensions
        data = np.zeros((height, width))

        # 3) Fill in cluster nodes with some mid-level intensity
        for node in cluster_nodes:
          r = int(G.nodes[node]['row'] - min_row)
          c = int(G.nodes[node]['column'] - min_col)
          data[r, c] = 0.5


        # 4) Mark the seed at a higher intensity
        seed_r = int(G.nodes[seed_id]['row'] - min_row)
        seed_c = int(G.nodes[seed_id]['column'] - min_col)
        data[seed_r, seed_c] = 1.0


        rows = [G.nodes[node]['row'] for node in cluster_nodes]
        cols = [G.nodes[node]['column'] for node in cluster_nodes]

        plt.scatter(cols, rows, c='blue', marker='s')
        # highlight seed in red


    return cluster_analysis, clustered_nodes


In [ ]:
def process_event(particles, evtnum, energy_threshold=50, window_size=5):
    seeds = find_seeds_optimized(particles, energy_threshold=energy_threshold)
    if not seeds:
        return None

    clusters = extract_fixed_window_clusters(particles, seeds, window_size)

    cluster_analysis = []
    clustered_nodes = {}

    for cluster_id, cluster in enumerate(clusters):
        seed_id = cluster['seed_id']
        seed_row = cluster['seed_row']
        seed_col = cluster['seed_col']
        mcid = cluster['mcid']
        energy = cluster['total_energy']

        global_cluster_id = f"{evtnum}_{cluster_id}"

        cluster_analysis.append({
            'Cluster ID': global_cluster_id,
            'Seed ID': seed_id,
            'Seed Row': seed_row,
            'Seed Column': seed_col,
            'Seed MCID': mcid,
            'Total Energy': energy
        })

        grid = cluster['normalized_grid']
        half_w = window_size // 2
        for r in range(window_size):
            for c in range(window_size):
                if grid[r, c] > 0:
                    abs_r = seed_row - half_w + r
                    abs_c = seed_col - half_w + c
                    node_id = f"{evtnum}_{cluster_id}_{r}_{c}"
                    clustered_nodes[node_id] = {
                        'node_id': node_id,
                        'row': abs_r,
                        'column': abs_c,
                        'energy': grid[r, c] * energy,
                        'MCID': mcid,
                        'type': 'neighbor' if (r, c) != (half_w, half_w) else 'seed',
                        'clusterId': global_cluster_id
                    }

    return cluster_analysis, clustered_nodes


In [ ]:
import numpy as np

def extract_fixed_window_clusters(particles, seeds, window_size=5):
    half_w = window_size // 2
    clusters = []

    for seed_id, sr, sc, seed_energy, mcid in seeds:
        # Extract 5x5 window around seed
        window_particles = particles[
            (particles['digitR'] >= sr - half_w) & (particles['digitR'] <= sr + half_w) &
            (particles['digitC'] >= sc - half_w) & (particles['digitC'] <= sc + half_w)
        ]

        # Create energy grid
        grid = np.zeros((window_size, window_size))
        for _, row in window_particles.iterrows():
          try:
              # Clean integer coordinates
              r_raw, c_raw = row['digitR'], row['digitC']
              if pd.isna(r_raw) or pd.isna(c_raw):
                  continue  # Skip if any are NaN

              r = int(round(r_raw)) - (sr - half_w)
              c = int(round(c_raw)) - (sc - half_w)

              if 0 <= r < window_size and 0 <= c < window_size:
                  grid[int(r), int(c)] = row['digitE']
          except Exception as e:
              print(f"Skipping row due to error: {e}")
              continue


        total_energy = grid.sum()
        if total_energy == 0:
            continue  # skip empty cluster

        # Normalize
        norm_grid = grid / total_energy

        # Optional: rotate based on max neighbor (e.g. north or northwest)
        # TODO: implement if you want canonical alignment

        clusters.append({
            'seed_id': seed_id,
            'seed_row': sr,
            'seed_col': sc,
            'mcid': mcid,
            'total_energy': total_energy,
            'normalized_grid': norm_grid
        })

    return clusters


In [ ]:
def process_top_n_events(filtered_df, top_n=100, energy_threshold=50):
    photon_counts_per_event = filtered_df[filtered_df["MCID"] == 22].groupby("evtnum").size().reset_index(name="Photon Count")
    top_events = photon_counts_per_event.sort_values(by="Photon Count", ascending=False).head(top_n)["evtnum"]

    all_cluster_analysis = []
    all_clustered_nodes = {}

    for evtnum in top_events:
        event_particles = filtered_df[filtered_df["evtnum"] == evtnum]
        result = process_event(event_particles, evtnum, energy_threshold)
        if result is None:
            continue
        cluster_analysis, clustered_nodes = result
        for ca in cluster_analysis:
            ca['evtnum'] = evtnum
            all_cluster_analysis.append(ca)
        for node in clustered_nodes.values():
            node['evtnum'] = evtnum
            all_clustered_nodes[node['node_id']] = node

    return all_cluster_analysis, all_clustered_nodes



In [ ]:
def process_all_events_fixed_clusters(df, energy_threshold=50, window_size=5):
    all_clusters = []

    for event_id in df['evtnum'].unique():
        event_particles = df[df['evtnum'] == event_id]
        event_particles = event_particles.reset_index(drop=True)

        seeds = find_seeds_optimized(event_particles, energy_threshold)
        if not seeds:
            continue

        clusters = extract_fixed_window_clusters(event_particles, seeds, window_size)
        all_clusters.extend(clusters)

    return all_clusters


In [ ]:
clusters = process_all_events_fixed_clusters(filtered_df, energy_threshold=50, window_size=5)



In [ ]:
print(f"Extracted {len(clusters)} clusters from {filtered_df['evtnum'].nunique()} events")


Extracted 49323 clusters from 3538 events


In [ ]:
def get_cluster_sizes(clusters):
    cluster_data = []

    for i, cluster in enumerate(clusters):
        # Count how many elements in the 5x5 normalized grid are > 0
        if 'normalized_grid' in cluster:
            size = (cluster['normalized_grid'] > 0).sum()
        else:
            size = None  # fallback if format is unexpected

        cluster_data.append({'clusterId': i, 'size': size})

    return pd.DataFrame(cluster_data)


In [ ]:
cluster_sizes_df = get_cluster_sizes(clusters)
print(cluster_sizes_df)


       clusterId  size
0              0     5
1              1     5
2              2     8
3              3    11
4              4    19
...          ...   ...
49318      49318     8
49319      49319     5
49320      49320     8
49321      49321    12
49322      49322     5

[49323 rows x 2 columns]


In [ ]:

if __name__ == '__main__':
    # Assume filtered_df is already defined, e.g. loaded from CSV.
    # filtered_df = pd.read_csv("your_filtered_data.csv")
                                                                              # before i had 50
    cluster_analysis_list, clustered_nodes_dict = process_top_n_events(filtered_df, top_n=2000, energy_threshold=50)


    df_nodes = pd.DataFrame(list(clustered_nodes_dict.values()))
    df_nodes.to_csv("nodes_with_cluster_ids_top10_events_optimized.csv", index=False)
    print("Saved node-cluster mapping to nodes_with_cluster_ids_top10_events_optimized.csv")

    df_clusters = pd.DataFrame(cluster_analysis_list)
    df_clusters.to_csv("clusters_photon_energy_top10_events_optimized.csv", index=False)
    print("Saved clusters data to clusters_photon_energy_top10_events_optimized.csv")



Saved node-cluster mapping to nodes_with_cluster_ids_top10_events_optimized.csv
Saved clusters data to clusters_photon_energy_top10_events_optimized.csv


In [ ]:
    df_nodes = pd.DataFrame(list(clustered_nodes_dict.values()))
    df_clusters = pd.DataFrame(cluster_analysis_list)

    # Save with consistent, descriptive filenames
    nodes_filename = "nodes_with_cluster_ids_top2000_events_fixedwindow.csv"
    clusters_filename = "clusters_photon_energy_top2000_events_fixedwindow.csv"

    df_nodes.to_csv(nodes_filename, index=False)
    print(f"Saved node-cluster mapping to {nodes_filename}")

    df_clusters.to_csv(clusters_filename, index=False)
    print(f"Saved clusters data to {clusters_filename}")

Saved node-cluster mapping to nodes_with_cluster_ids_top2000_events_fixedwindow.csv
Saved clusters data to clusters_photon_energy_top2000_events_fixedwindow.csv


In [ ]:
print(f"clusters: {len(clusters)}")
print(f"df_clusters: {len(df_clusters)}")
print(f"df_nodes: {len(df_nodes)}")
print(f"clusterIds in df_nodes: {df_nodes['clusterId'].nunique()}")

clusters: 49323
df_clusters: 35503
df_nodes: 406788
clusterIds in df_nodes: 35503


In [ ]:
len(df_clusters)

35503

In [ ]:
len(df_nodes)

406788

In [ ]:
df_clusters

,Cluster ID,Seed ID,Seed Row,Seed Column,Seed MCID,Total Energy,evtnum
0,0,"(1439, 2, 29)",13,0,-321.0,753.412015,42367
1,1,"(1439, 10, 29)",15,2,-211.0,1701.242006,42367
2,2,"(1439, 24, 29)",17,6,-321.0,5178.858111,42367
3,3,"(1439, 37, 29)",19,4,22.0,4865.396086,42367
4,4,"(1439, 42, 29)",20,1,-211.0,1499.630007,42367
...,...,...,...,...,...,...,...
35498,8,"(100, 73, 29)",33,0,-211.0,658.588015,252735
35499,9,"(100, 84, 29)",42,5,-2212.0,1661.846039,252735
35500,10,"(100, 91, 29)",44,4,-2212.0,1656.402037,252735
35501,11,"(100, 100, 29)",48,5,22.0,378.603997,252735
